In [1]:
from flask import Flask
 
from flask_apscheduler import APScheduler
 
import datetime
 


In [3]:
import sys
sys.path.append('E:\wangzhilin\QuantumGalaxy\server_new')
from utils import get_mysql,get_logger
import logging
logger=get_logger(__name__,chlevel=logging.INFO)


In [5]:
mysql=get_mysql('web_server')
mysql.read_query('select * from ev_band_info')

(('000525.SZ', '000525.SZ ST红太阳', 1),
 ('002518.SZ', '002518.SZ 科士达', 1),
 ('002543.SZ', '002543.SZ 万和电气', 1),
 ('002896.SZ', '002896.SZ 中大力德', 1),
 ('003023.SZ', '003023.SZ 彩虹集团', 1),
 ('1164.HK', '1164.HK 中广核矿业', 1),
 ('1797.HK', '1797.HK 新东方在线', 1),
 ('300109.SZ', '300109.SZ 新开源', 1),
 ('300438.SZ', '300438.SZ 鹏辉能源', 1),
 ('300700.SZ', '300700.SZ 岱勒新材', 1),
 ('300881.SZ', '300881.SZ 盛德鑫泰', 1),
 ('600011.SH', '600011.SH 华能国际', 1),
 ('600026.SH', '600026.SH 中远海能', 1),
 ('600027.SH', '600027.SH 华电国际', 1),
 ('600230.SH', '600230.SH 沧州大化', 1),
 ('600536.SH', '600536.SH 中国软件', 1),
 ('600578.SH', '600578.SH 京能电力', 1),
 ('600603.SH', '600603.SH 广汇物流', 1),
 ('600795.SH', '600795.SH 国电电力', 1),
 ('600977.SH', '600977.SH 中国电影', 1),
 ('601975.SH', '601975.SH 招商南油', 1),
 ('603366.SH', '603366.SH 日出东方', 1),
 ('603477.SH', '603477.SH 巨星农牧', 1),
 ('603728.SH', '603728.SH 鸣志电器', 1),
 ('603757.SH', '603757.SH 大元泵业', 1),
 ('605333.SH', '605333.SH 沪光股份', 1),
 ('688063.SH', '688063.SH 派能科技', 1),
 ('68821

In [15]:


from blueprints.ev_band.models import Code2JS


In [18]:
from atlassian import Jira
mysql=get_mysql(database='web_server')
jira = Jira(
    url='https://research.quantumgalaxy.cn/',
    username='wangzhilin',
    password='wangzhilin')

{'start_date': datetime.datetime(2022, 9, 5, 23, 27),
 'target_date': datetime.datetime(2022, 12, 31, 0, 0),
 'pc': '0.7',
 'xc': '18',
 'xo': '35',
 'po': '0.3'}

In [8]:
def reset_mysql():

#print(self.code)

    issues=jira.jql('project = POSITION and cf[11723] in(胸有成竹,应该能用,不太确定)')
    issues=issues['issues']
    r=[] 
    for issue in issues:
        #print(issue)
        code=issue['fields']['customfield_10201']
        summary=issue['fields']['summary']
        r.append((code,summary))
    mysql.write_query('update ev_band_info set status=0')
    mysql.write_many_query(sql='insert into ev_band_info (code,name,status) values (%s,%s,1) ON DUPLICATE KEY UPDATE status=1',val=r)

In [37]:
def fetch_data():
    codes=mysql.read_query('select code from ev_band_info where status=1')
    for c in codes:
        c=c[0]
        worker=Code2JS(c)
        history=worker.split_ev_band(worker.clean_data(worker.find_issue()))
        for h in history:
            mysql.write_query("insert ignore into ev_band_data (code,start_date,target_date,pc,xc,po,xo) values ('%s','%s','%s',%s,%s,%s,%s)"%(c,*(h.values()) ))
    

In [39]:
history

[{'start_date': datetime.datetime(2022, 9, 5, 23, 27),
  'target_date': datetime.datetime(2022, 12, 31, 0, 0),
  'pc': '0.7',
  'xc': '18',
  'xo': '35',
  'po': '0.3'},
 {'start_date': datetime.datetime(2022, 9, 20, 2, 27),
  'target_date': datetime.datetime(2023, 4, 30, 0, 0),
  'pc': '0.8',
  'xc': '18',
  'xo': '38',
  'po': '0.4'},
 {'start_date': datetime.datetime(2022, 10, 18, 10, 52),
  'target_date': datetime.datetime(2023, 3, 31, 0, 0),
  'pc': '0.8',
  'xc': '37',
  'xo': '60',
  'po': '0.2'}]

In [42]:
worker.code

'YANG.P'

In [43]:
history=worker.split_ev_band(worker.clean_data(worker.find_issue()))

In [44]:
history

[{'start_date': datetime.datetime(2022, 9, 5, 23, 27),
  'target_date': datetime.datetime(2022, 12, 31, 0, 0),
  'pc': '0.7',
  'xc': '18',
  'xo': '35',
  'po': '0.3'},
 {'start_date': datetime.datetime(2022, 9, 20, 2, 27),
  'target_date': datetime.datetime(2023, 4, 30, 0, 0),
  'pc': '0.8',
  'xc': '18',
  'xo': '38',
  'po': '0.4'},
 {'start_date': datetime.datetime(2022, 10, 18, 10, 52),
  'target_date': datetime.datetime(2023, 3, 31, 0, 0),
  'pc': '0.8',
  'xc': '37',
  'xo': '60',
  'po': '0.2'}]

In [51]:
code='YANG.P'

if mysql.read_query('select status from ev_band_info where code="%s"'%code)[0][0]:
    res=mysql.read_query('select start_date,target_date,pc,xc,xo,po from ev_band_data where code="%s"'%code)

In [54]:
dict(zip(('start_date','target_date','pc','xc','xo','po'),res[0]))

{'start_date': datetime.date(2022, 9, 5),
 'target_date': datetime.date(2022, 12, 31),
 'pc': 0.7,
 'xc': 18.0,
 'xo': 0.3,
 'po': 35.0}

In [56]:
h=[]
for r in res:
    h.append(dict(zip(('start_date','target_date','pc','xc','xo','po'),r)))
h

[{'start_date': datetime.date(2022, 9, 5),
  'target_date': datetime.date(2022, 12, 31),
  'pc': 0.7,
  'xc': 18.0,
  'xo': 0.3,
  'po': 35.0},
 {'start_date': datetime.date(2022, 9, 20),
  'target_date': datetime.date(2023, 4, 30),
  'pc': 0.8,
  'xc': 18.0,
  'xo': 0.4,
  'po': 38.0},
 {'start_date': datetime.date(2022, 10, 18),
  'target_date': datetime.date(2023, 3, 31),
  'pc': 0.8,
  'xc': 37.0,
  'xo': 0.2,
  'po': 60.0}]

In [58]:
history

[{'start_date': datetime.datetime(2022, 9, 5, 23, 27),
  'target_date': datetime.datetime(2022, 12, 31, 0, 0),
  'pc': '0.7',
  'xc': '18',
  'xo': '35',
  'po': '0.3'},
 {'start_date': datetime.datetime(2022, 9, 20, 2, 27),
  'target_date': datetime.datetime(2023, 4, 30, 0, 0),
  'pc': '0.8',
  'xc': '18',
  'xo': '38',
  'po': '0.4'},
 {'start_date': datetime.datetime(2022, 10, 18, 10, 52),
  'target_date': datetime.datetime(2023, 3, 31, 0, 0),
  'pc': '0.8',
  'xc': '37',
  'xo': '60',
  'po': '0.2'}]

In [59]:
a=[{'start_date': datetime.date(2022, 8, 25), 'target_date': datetime.date(2023, 4, 30), 'pc': 0.7, 'xc': 13.0, 'xo': 0.4, 'po': 54.0}, {'start_date': datetime.date(2022, 8, 26), 'target_date': datetime.date(2023, 4, 30), 'pc': 0.7, 'xc': 13.0, 'xo': 0.3, 'po': 54.0}, {'start_date': datetime.date(2022, 9, 15), 'target_date': datetime.date(2023, 4, 30), 'pc': 0.7, 'xc': 6.0, 'xo': 0.3, 'po': 54.0}, {'start_date': datetime.date(2022, 10, 3), 'target_date': datetime.date(2023, 4, 30), 'pc': 0.6, 'xc': 12.0, 'xo': 0.3, 'po': 41.4}]


In [60]:
b=[{'start_date': datetime.datetime(2022, 8, 25, 22, 34), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': '0.7', 'xc': '13', 'xo': '54', 'po': '0.4'}, {'start_date': datetime.datetime(2022, 8, 26, 11, 1), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': '0.7', 'xc': '13', 'xo': '54', 'po': '0.3'}, {'start_date': datetime.datetime(2022, 9, 15, 13, 50), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': '0.7', 'xc': '6', 'xo': '54', 'po': '0.3'}, {'start_date': datetime.datetime(2022, 10, 3, 21, 59), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': '0.6', 'xc': '12', 'xo': '41.4', 'po': '0.3'}]

In [64]:
a[0]

{'start_date': datetime.date(2022, 8, 25),
 'target_date': datetime.date(2023, 4, 30),
 'pc': 0.7,
 'xc': 13.0,
 'xo': 0.4,
 'po': 54.0}

In [65]:
b[0]

{'start_date': datetime.datetime(2022, 8, 25, 22, 34),
 'target_date': datetime.datetime(2023, 4, 30, 0, 0),
 'pc': '0.7',
 'xc': '13',
 'xo': '54',
 'po': '0.4'}

In [ ]:
0.4 13.0 54.0 0.7 2023-04-30 00:00:00 2022-08-25 22:34:00
54.0 13.0 0.4 0.7 2023-04-30 00:00:00 2022-08-25 22:34:00

In [66]:
a=[{'start_date': datetime.datetime(2022, 8, 25, 22, 34), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': '0.7', 'xc': '13', 'xo': '54', 'po': '0.4'}, {'start_date': datetime.datetime(2022, 8, 26, 11, 1), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': '0.7', 'xc': '13', 'xo': '54', 'po': '0.3'}, {'start_date': datetime.datetime(2022, 9, 15, 13, 50), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': '0.7', 'xc': '6', 'xo': '54', 'po': '0.3'}, {'start_date': datetime.datetime(2022, 10, 3, 21, 59), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': '0.6', 'xc': '12', 'xo': '41.4', 'po': '0.3'}]
b=[{'start_date': datetime.datetime(2022, 8, 25, 22, 34), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': 0.7, 'xc': 13.0, 'xo': 0.4, 'po': 54.0}, {'start_date': datetime.datetime(2022, 8, 26, 11, 1), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': 0.7, 'xc': 13.0, 'xo': 0.3, 'po': 54.0}, {'start_date': datetime.datetime(2022, 9, 15, 13, 50), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': 0.7, 'xc': 6.0, 'xo': 0.3, 'po': 54.0}, {'start_date': datetime.datetime(2022, 10, 3, 21, 59), 'target_date': datetime.datetime(2023, 4, 30, 0, 0), 'pc': 0.6, 'xc': 12.0, 'xo': 0.3, 'po': 41.4}]


In [67]:
a[0]

{'start_date': datetime.datetime(2022, 8, 25, 22, 34),
 'target_date': datetime.datetime(2023, 4, 30, 0, 0),
 'pc': '0.7',
 'xc': '13',
 'xo': '54',
 'po': '0.4'}

In [68]:
b[0]

{'start_date': datetime.datetime(2022, 8, 25, 22, 34),
 'target_date': datetime.datetime(2023, 4, 30, 0, 0),
 'pc': 0.7,
 'xc': 13.0,
 'xo': 0.4,
 'po': 54.0}

In [72]:
history[0]

{'start_date': datetime.datetime(2022, 9, 5, 23, 27),
 'target_date': datetime.datetime(2022, 12, 31, 0, 0),
 'pc': '0.7',
 'xc': '18',
 'xo': '35',
 'po': '0.3'}